# Drive-Time Isochrones using Valhalla

Uses Valhalla API for real road network routing and isochrone generation.
- **100% Free & Open Source**
- **Native isochrone support** - faster and more accurate than OSRM
- **No installation needed** - uses public Valhalla server
- **No API key required**

**API Reference:** https://valhalla.github.io/valhalla/api/isochrone/api-reference/

## Parameters

In [ ]:
import requests
import json
from pyspark.sql.functions import col, expr, lit

dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("gold_schema", "")
dbutils.widgets.text("current_stores_table", "current_stores_ne")
dbutils.widgets.dropdown("input_source", "lce", ["lce", "partners", "candidate"], "Input Source")
dbutils.widgets.text("valhalla_url", "https://valhalla1.openstreetmap.de")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
current_stores_table = dbutils.widgets.get("current_stores_table")
input_source = dbutils.widgets.get("input_source")
valhalla_url = dbutils.widgets.get("valhalla_url")

# Determine input and output tables based on source
if input_source == "lce":
    input_table = f"{catalog}.{bronze_schema}.{current_stores_table}"
    output_table = f"{catalog}.{silver_schema}.isochrones_lce"
    store_type_label = "Current Store"
elif input_source == "partners":
    input_table = f"{catalog}.{silver_schema}.pois_partners"
    output_table = f"{catalog}.{silver_schema}.isochrones_partners"
    store_type_label = "Partner Store"
elif input_source == "candidate":
    input_table = f"{catalog}.{silver_schema}.whitespace_locations"
    output_table = f"{catalog}.{silver_schema}.candidate_isochrones"
    store_type_label = "Expansion Candidate"
else:
    raise ValueError(f"Unknown input_source: {input_source}")

print(f"Input source: {input_source}")
print(f"Input table: {input_table}")
print(f"Output table: {output_table}")
print(f"Valhalla URL: {valhalla_url}")

## Test Valhalla Connection

In [ ]:
import time

# Test Valhalla API with a sample location (Boston)
test_payload = {
    "locations": [{"lat": 42.3601, "lon": -71.0589}],
    "costing": "auto",
    "contours": [{"time": 5}],
    "polygons": True
}

test_url = f"{valhalla_url}/isochrone"

try:
    response = requests.post(test_url, json=test_payload, timeout=30)
    response.raise_for_status()
    test_result = response.json()
    
    print("✓ Successfully connected to Valhalla API")
    print(f"  Server: {valhalla_url}")
    print(f"  Test response features: {len(test_result.get('features', []))}")
except Exception as e:
    print(f"❌ Could not connect to Valhalla API: {e}")
    raise

time.sleep(0.5)

## Load Locations

In [ ]:
# Read locations from the appropriate table
locations = spark.table(input_table)

# Auto-detect columns (handles both LCE and POI schemas)
columns = locations.columns
id_col = next((c for c in columns if c in ['store_number', 'point_id', 'id', 'location_id', 'poi_id']), columns[0])
lat_col = next((c for c in columns if c in ['latitude', 'lat', 'y']), None)
lon_col = next((c for c in columns if c in ['longitude', 'lon', 'lng', 'x']), None)
city_col = next((c for c in columns if c in ['city', 'municipality']), None)
state_col = next((c for c in columns if c in ['state', 'region', 'state_abbr']), None)
name_col = next((c for c in columns if c in ['name', 'store_name']), None)
address_col = next((c for c in columns if c in ['address', 'full_address', 'addr']), None)

if not lat_col or not lon_col:
    raise ValueError(f"Cannot find lat/lon columns. Available: {columns}")

# Standardize columns
locations_std = locations.select(
    col(id_col).alias("location_id"),
    col(lat_col).cast("double").alias("latitude"),
    col(lon_col).cast("double").alias("longitude"),
    (col(name_col) if name_col else lit(store_type_label)).alias("store_type"),
    (col(city_col) if city_col else lit(None)).alias("city"),
    (col(state_col) if state_col else lit(None)).alias("state"),
    (col(address_col) if address_col else lit(None)).alias("address")
).filter(col("latitude").isNotNull() & col("longitude").isNotNull())

# Fixed 5-minute drive time
locations_with_times = locations_std.withColumn(
    "drive_time_minutes", lit(5)
)

print(f"Source: {input_source}")
print(f"Loaded {locations_with_times.count()} locations from {input_table}")
print(f"Using fixed 5-minute drive time")
display(locations_with_times.limit(5))

## Generate Isochrones with Valhalla

In [ ]:
import time

def get_valhalla_isochrone(lat, lon, minutes, valhalla_url):
    """
    Get isochrone from Valhalla API
    Returns: WKT polygon string
    """
    payload = {
        "locations": [{"lat": lat, "lon": lon}],
        "costing": "auto",
        "contours": [{"time": minutes}],
        "polygons": True
    }
    
    try:
        response = requests.post(
            f"{valhalla_url}/isochrone",
            json=payload,
            timeout=30
        )
        response.raise_for_status()
        result = response.json()
        
        # Extract polygon from GeoJSON response
        if result.get('features') and len(result['features']) > 0:
            feature = result['features'][0]
            geometry = feature.get('geometry')
            
            if geometry and geometry.get('type') == 'Polygon':
                coords = geometry['coordinates'][0]  # Exterior ring
                # Convert to WKT: POLYGON ((lon lat, lon lat, ...))
                coords_str = ', '.join([f"{lon} {lat}" for lon, lat in coords])
                return f"POLYGON (({coords_str}))"
        
        return None
        
    except Exception as e:
        # Silently skip errors for individual locations
        return None

# Test with first location
test_location = locations_with_times.first()
print(f"Testing isochrone generation for: {test_location.location_id}")
print(f"Location: ({test_location.latitude}, {test_location.longitude})")

test_wkt = get_valhalla_isochrone(
    test_location.latitude, 
    test_location.longitude, 
    5, 
    valhalla_url
)

if test_wkt:
    print("\n✓ Test isochrone generated successfully!")
    print(f"  WKT length: {len(test_wkt)} characters")
else:
    print("\n⚠ Test failed - no isochrone generated")

## Generate All Isochrones

In [ ]:
from pyspark.sql import Row
import time

location_rows = locations_with_times.collect()
print(f"Generating 5-minute isochrones for {len(location_rows)} locations...")

results = []
start_time = time.time()

for i, row in enumerate(location_rows):
    if i % 5 == 0 and i > 0:
        elapsed = time.time() - start_time
        avg_time = elapsed / i
        remaining = (len(location_rows) - i) * avg_time
        print(f"Progress: {i}/{len(location_rows)} ({i/len(location_rows)*100:.1f}%) - "
              f"Elapsed: {elapsed:.1f}s - ETA: {remaining:.1f}s")
    
    wkt = get_valhalla_isochrone(
        row.latitude,
        row.longitude,
        row.drive_time_minutes,
        valhalla_url
    )
    
    if wkt:
        results.append(Row(
            location_id=row.location_id,
            latitude=row.latitude,
            longitude=row.longitude,
            store_type=row.store_type,
            city=row.city,
            state=row.state,
            address=row.address,
            drive_time_minutes=row.drive_time_minutes,
            geometry_wkt=wkt
        ))
    
    # Small delay to be respectful of the API
    time.sleep(0.1)

total_time = time.time() - start_time
print(f"✅ Generated {len(results)}/{len(location_rows)} isochrones successfully")
print(f"   Total time: {total_time:.1f} seconds")
print(f"   Average: {total_time/len(location_rows):.2f} seconds per location")

## Save to Delta

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import current_timestamp

# Create DataFrame
isochrone_schema = StructType([
    StructField("location_id", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("store_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("address", StringType(), True),
    StructField("drive_time_minutes", IntegerType(), False),
    StructField("geometry_wkt", StringType(), False)
])

isochrones_df = spark.createDataFrame(results, schema=isochrone_schema)

# Convert to geometry and add metadata
isochrones_final = (
    isochrones_df
    .withColumn("geometry", expr("ST_GeomFromText(geometry_wkt, 4326)"))
    .withColumn("area_sqkm", expr("ST_Area(geometry) / 1000000"))
    .withColumn("created_timestamp", current_timestamp())
    .withColumn("routing_provider", lit("valhalla"))
    .drop("geometry_wkt")
)

# Write to Delta
(
    isochrones_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Saved {len(results)} isochrones to {output_table}")

## Summary

In [ ]:
display(spark.sql(f"""
    SELECT
        routing_provider,
        drive_time_minutes,
        COUNT(*) as count,
        ROUND(AVG(area_sqkm), 2) as avg_area_sqkm,
        ROUND(MIN(area_sqkm), 2) as min_area_sqkm,
        ROUND(MAX(area_sqkm), 2) as max_area_sqkm
    FROM {output_table}
    GROUP BY routing_provider, drive_time_minutes
    ORDER BY routing_provider
"""))

## Visualize with Folium

In [ ]:
import folium
from shapely import wkt as shapely_wkt

# Create map centered on Massachusetts
ma_center = [42.4072, -71.3824]
m = folium.Map(location=ma_center, zoom_start=8, tiles='OpenStreetMap')

# Add isochrone polygons
print(f"Adding {len(results)} isochrone polygons...")
for result in results:
    polygon = shapely_wkt.loads(result.geometry_wkt)
    coords = [[lat, lon] for lon, lat in polygon.exterior.coords]
    
    folium.Polygon(
        locations=coords,
        color='#FF6B35',
        fillColor='#FF6B35',
        fillOpacity=0.2,
        weight=2,
        popup=f"{result.store_type}<br>Store: {result.location_id}<br>5 min drive time (Valhalla)"
    ).add_to(m)

# Add location markers
print(f"Adding {len(location_rows)} location markers...")
for location in location_rows:
    folium.CircleMarker(
        location=[location.latitude, location.longitude],
        radius=6,
        popup=f"<b>{location.store_type}</b><br>ID: {location.location_id}",
        color='#C1121F',
        fillColor='#C1121F',
        fillOpacity=0.8,
        weight=2
    ).add_to(m)

# Add legend
legend_html = f'''
<div style="position: fixed; 
            bottom: 50px; right: 50px; width: 200px; height: 100px; 
            background-color: white; border:2px solid grey; z-index:9999; 
            font-size:14px; padding: 10px">
<p style="margin-bottom: 5px;"><b>Legend</b></p>
<p style="margin: 5px 0;"><span style="color: #C1121F;">●</span> {store_type_label}</p>
<p style="margin: 5px 0;"><span style="background-color: rgba(255,107,53,0.3); padding: 0 8px;">█</span> 5-min Drive (Valhalla)</p>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"\n✅ Map created with {len(results)} isochrones and {len(location_rows)} locations")
m